In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from bs4 import BeautifulSoup

# Load the CSV file into a DataFrame
df = pd.read_csv('postings.csv')

# Display the first 5 rows to get a quick overview of the data
print("First 5 rows of the DataFrame:")
display(df.head())

First 5 rows of the DataFrame:


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0


In [3]:
# --- CLEANING PHASE ---


def clean_linkedin_data(df):
    print(f"Original shape: {df.shape}")
    
    # --- 1. ID & ENTITY CLEANING ---
    # Xóa dòng trùng lặp job_id
    df = df.drop_duplicates(subset=['job_id'])
    
    # Xử lý Company ID (Float -> Int -> Int cho phép Null)
    df['company_id'] = df['company_id'].astype('Int64')
    
    # Fill tên công ty thiếu
    df['company_name'] = df['company_name'].fillna('Unknown Company')

    # --- 2. TEMPORAL CLEANING (Thời gian) ---
    time_cols = ['original_listed_time', 'listed_time', 'expiry', 'closed_time']
    for col in time_cols:
        df[col] = pd.to_datetime(df[col], unit='ms')
    
    # Logic check: Nếu closed_time < listed_time thì set closed_time là NaT (Lỗi dữ liệu)
    mask_invalid_time = df['closed_time'] < df['listed_time']
    df.loc[mask_invalid_time, 'closed_time'] = pd.NaT

    # --- 3. FINANCIAL CLEANING (Lương) ---
    # Quy đổi hết về Lương Năm (Annual Salary)
    def calculate_annual_salary(row):
        # Xác định mức lương cơ sở
        if pd.notna(row['med_salary']):
            base = row['med_salary']
        elif pd.notna(row['min_salary']) and pd.notna(row['max_salary']):
            base = (row['min_salary'] + row['max_salary']) / 2
        elif pd.notna(row['max_salary']):
            base = row['max_salary']
        else:
            return np.nan # Không có thông tin lương

        # Quy đổi dựa trên pay_period
        period = str(row['pay_period']).upper()
        if period == 'HOURLY': return base * 2080
        if period == 'MONTHLY': return base * 12
        if period == 'WEEKLY': return base * 52
        if period == 'YEARLY': return base
        return base # Mặc định giữ nguyên nếu không rõ (hoặc return NaN tùy độ strict)

    df['standardized_annual_salary'] = df.apply(calculate_annual_salary, axis=1)

    # --- 4. CATEGORICAL & METADATA CLEANING ---
    # Remote: NaN -> 0 (False), 1.0 -> 1 (True)
    df['remote_allowed'] = df['remote_allowed'].fillna(0).astype(int)
    
    # Sponsored: Đảm bảo là 0/1
    df['sponsored'] = df['sponsored'].astype(int)
    
    # Experience Level: Fill NaN
    df['formatted_experience_level'] = df['formatted_experience_level'].fillna('Unknown')
    
    # Views & Applies: NaN -> 0 (Giả định chưa có lượt xem/nộp)
    df['views'] = df['views'].fillna(0).astype(int)
    df['applies'] = df['applies'].fillna(0).astype(int)

    # --- 5. TEXT CLEANING (Nâng cao) ---
    def clean_text_advanced(text):
        if pd.isna(text): return ""
        # 1. Bỏ HTML tags
        soup = BeautifulSoup(text, "html.parser")
        text = soup.get_text(separator=" ")
        # 2. Bỏ URLs
        text = re.sub(r'http\S+', '', text)
        # 3. Bỏ Emails
        text = re.sub(r'\S+@\S+', '', text)
        # 4. Bỏ nhiều khoảng trắng thừa
        text = re.sub(r'\s+', ' ', text).strip()
        # 5. Lowercase
        return text.lower()

    df['clean_description'] = df['description'].apply(clean_text_advanced)
    
    # --- 6. GEOGRAPHY ---
    # Tách State từ Location (Định dạng: City, State)
    df['job_state'] = df['location'].apply(lambda x: x.split(',')[-1].strip() if ',' in x else 'Unknown')
    # Zip code: Chuyển về string, bỏ .0
    df['zip_code'] = df['zip_code'].astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', np.nan)

    print(f"Cleaned shape: {df.shape}")
    return df

# Chạy function
df_clean = clean_linkedin_data(df)

# Kiểm tra kết quả
print(df_clean[['job_id', 'standardized_annual_salary', 'remote_allowed', 'job_state']].head())
print(df_clean.info())


Original shape: (123849, 31)


/var/folders/qw/15rb_9bn0_3fqr7dgvy560g40000gn/T/ipykernel_28050/1633432408.py:67: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  soup = BeautifulSoup(text, "html.parser")


Cleaned shape: (123849, 34)
     job_id  standardized_annual_salary  remote_allowed job_state
0    921716                     38480.0               0        NJ
1   1829192                     83200.0               0        CO
2  10998357                     55000.0               0        OH
3  23221523                    157500.0               0        NY
4  35982263                     70000.0               0        IA
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 34 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   job_id                      123849 non-null  int64         
 1   company_name                123849 non-null  object        
 2   title                       123849 non-null  object        
 3   description                 123842 non-null  object        
 4   max_salary                  29793 non-null   float64       
 5   pay_p

In [ ]:
# --- FILTERING COLUMN PHASE ---
# Chọn các cột giữ lại (Final Features)
selected_columns = [
    # --- Identity ---
    'job_id', 'company_id', 'company_name',
    
    # --- Content ---
    'title', 'clean_description', # Dùng cái đã clean
    
    # --- Logistics & Location ---
    'job_state', 'location', 'remote_allowed', 'formatted_work_type',
    
    # --- Financials ---
    'standardized_annual_salary', # Dùng cái đã chuẩn hóa
    
    # --- Time ---
    'listed_time', 'expiry', 'closed_time',
    
    # --- Metadata (Optional - Giữ lại để phân tích độ hot) ---
    'views', 'applies', 'sponsored', 'formatted_experience_level'
]

# Tạo DataFrame cuối cùng
final_df = df_clean[selected_columns]

# Kết quả: Số cột sẽ giảm xuống gọn gàng (khoảng 18-20 cột)
print(f"Final shape: {final_df.shape}")

# Lưu file
final_df.to_csv('cleaned_postings.csv', index=False)

Final shape: (123849, 17)


In [6]:
final_df

,job_id,company_id,company_name,title,clean_description,job_state,location,remote_allowed,formatted_work_type,standardized_annual_salary,listed_time,expiry,closed_time,views,applies,sponsored,formatted_experience_level
0,921716,2774458,Corcoran Sawyer Smith,Marketing Coordinator,job descriptiona leading real estate firm in n...,NJ,"Princeton, NJ",0,Full-time,38480.0,2024-04-17 23:45:08,2024-05-17 23:45:08,NaT,20,2,0,Unknown
1,1829192,<NA>,Unknown Company,Mental Health Therapist/Counselor,"at aspen therapy and wellness , we are committ...",CO,"Fort Collins, CO",0,Full-time,83200.0,2024-04-11 17:51:27,2024-05-11 17:51:27,NaT,1,0,0,Unknown
2,10998357,64896719,The National Exemplar,Assitant Restaurant Manager,the national exemplar is accepting application...,OH,"Cincinnati, OH",0,Full-time,55000.0,2024-04-16 14:26:54,2024-05-16 14:26:54,NaT,8,0,0,Unknown
3,23221523,766262,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,senior associate attorney - elder law / trusts...,NY,"New Hyde Park, NY",0,Full-time,157500.0,2024-04-12 04:23:32,2024-05-12 04:23:32,NaT,16,0,0,Unknown
4,35982263,<NA>,Unknown Company,Service Technician,looking for hvac service tech with experience ...,IA,"Burlington, IA",0,Full-time,70000.0,2024-04-18 14:52:23,2024-05-18 14:52:23,NaT,3,0,0,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123844,3906267117,56120,Lozano Smith,Title IX/Investigations Attorney,our walnut creek office is currently seeking a...,CA,"Walnut Creek, CA",0,Full-time,157500.0,2024-04-20 00:00:23,2024-05-20 00:00:23,NaT,1,0,0,Mid-Senior level
123845,3906267126,1124131,Pinterest,"Staff Software Engineer, ML Serving Platform",about pinterest: millions of people across the...,Unknown,United States,1,Full-time,NaN,2024-04-20 00:17:16,2024-05-20 00:17:16,NaT,3,0,0,Mid-Senior level
123846,3906267131,90552133,EPS Learning,"Account Executive, Oregon/Washington",company overview eps learning is a leading k–1...,WA,"Spokane, WA",1,Full-time,NaN,2024-04-20 00:18:59,2024-05-20 00:18:59,NaT,3,0,0,Mid-Senior level
123847,3906267195,2793699,Trelleborg Applied Technologies,Business Development Manager,the business development manager is a 'hunter'...,United States,"Texas, United States",1,Full-time,NaN,2024-04-20 00:23:52,2024-05-20 00:23:52,NaT,4,0,0,Unknown
